<a href="https://colab.research.google.com/github/Emmanuel-Oyewole/bookie-buster/blob/data-preprocessing/Bookie_buster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bookie Buster: A Football Betting Value Predictor

This Jupyter Notebook outlines the development and training of a machine learning model named **Bookie Buster**. The goal of this project is to create an analytical tool that can predict football match outcomes and, more importantly, identify "value bets"—instances where the odds offered by bookmakers have a positive Expected Value (+EV).

The model leverages a comprehensive dataset, including historical match results, team statistics, and a full range of betting odds (opening and closing), to learn patterns and make informed predictions. By comparing its own probability estimations with the bookmakers' implied probabilities, Bookie Buster aims to provide a data-driven edge over the market.

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!ls /content/drive/My\ Drive/

 Classroom	    ml_dataset_linear_regression.xlsx  'Untitled document.gdoc'
'Colab Notebooks'  'NELFUND LOAN.gdoc'


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
file_path = '/content/drive/My Drive/ml_dataset_linear_regression.xlsx'

In [12]:
df = pd.read_excel(file_path)

In [17]:
df.head()

,Home Position,Away Position,Home Played,Away Played,Competition Progress,Is Friendly,Competition Type,League Predictability,Implied Odds,Opening Odds,Closing Odds,EV,beat_closing
0,7.0,9.0,28.0,30.0,84.0,No,League,high,1.94,1.59,1.56,-18.0991,0
1,3.0,2.0,27.0,27.0,99.0,No,League,medium,2.01,1.90,1.79,-5.5130,0
2,2.0,10.0,30.0,30.0,78.0,No,League,medium,2.20,2.21,2.24,0.4666,0
3,6.0,7.0,25.0,25.0,70.0,No,League,high,1.99,1.73,1.81,-13.1886,0
4,12.0,7.0,27.0,27.0,73.0,No,League,medium,1.81,1.67,1.60,-7.6824,0


In [18]:
# drop the EV column
df.drop(columns=["EV"], inplace=True)

In [19]:
df.head()

,Home Position,Away Position,Home Played,Away Played,Competition Progress,Is Friendly,Competition Type,League Predictability,Implied Odds,Opening Odds,Closing Odds,beat_closing
0,7.0,9.0,28.0,30.0,84.0,No,League,high,1.94,1.59,1.56,0
1,3.0,2.0,27.0,27.0,99.0,No,League,medium,2.01,1.90,1.79,0
2,2.0,10.0,30.0,30.0,78.0,No,League,medium,2.20,2.21,2.24,0
3,6.0,7.0,25.0,25.0,70.0,No,League,high,1.99,1.73,1.81,0
4,12.0,7.0,27.0,27.0,73.0,No,League,medium,1.81,1.67,1.60,0


In [22]:
# inspect missing attribute
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113493 entries, 0 to 113492
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Home Position          113485 non-null  float64
 1   Away Position          113485 non-null  float64
 2   Home Played            113165 non-null  float64
 3   Away Played            113165 non-null  float64
 4   Competition Progress   108165 non-null  float64
 5   Is Friendly            113493 non-null  object 
 6   Competition Type       113493 non-null  object 
 7   League Predictability  112903 non-null  object 
 8   Implied Odds           113493 non-null  float64
 9   Opening Odds           113493 non-null  float64
 10  Closing Odds           113493 non-null  float64
 11  beat_closing           113493 non-null  int64  
dtypes: float64(8), int64(1), object(3)
memory usage: 10.4+ MB


In [23]:
# percentage of missing values in each column
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage

,0
Home Position,0.007049
Away Position,0.007049
Home Played,0.289005
Away Played,0.289005
Competition Progress,4.694563
Is Friendly,0.000000
Competition Type,0.000000
League Predictability,0.519856
Implied Odds,0.000000
Opening Odds,0.000000


9.0

In [ ]:
# confirm if there are still null values for corners
df["Corners"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 113493 entries, 0 to 113492
Series name: Corners
Non-Null Count   Dtype  
--------------   -----  
113493 non-null  float64
dtypes: float64(1)
memory usage: 886.8 KB


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113493 entries, 0 to 113492
Data columns (total 24 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Kickoff                113493 non-null  datetime64[ns]
 1   Status                 113493 non-null  object        
 2   Home Goals             113493 non-null  int64         
 3   Away Goals             113493 non-null  int64         
 4   Corners                113493 non-null  float64       
 5   HT Score               113378 non-null  object        
 6   Home Position          113485 non-null  float64       
 7   Away Position          113485 non-null  float64       
 8   Home Played            113165 non-null  float64       
 9   Away Played            113165 non-null  float64       
 10  Competition Progress   108165 non-null  float64       
 11  Is Friendly            113493 non-null  object        
 12  Competition Type       113493 non-null  obje